
# Resolução de Problemas por Busca — Parte 1 e 2
**Engenharia da Computação / Eng. de Controle e Automação — IFMT**  
**Tema:** Busca não informada (BFS/DFS) e Busca informada (Gulosa, A*)  
**Bibliotecas:** `networkx`, `heapq` (padrão Python)  
    
> Este notebook foi preparado para aula prática. Use as células em ordem.  
> Versão: **GABARITO (com soluções)**.



## Pré-requisitos e instalação rápida
No **Colab**, execute a célula abaixo para instalar o `networkx` (se necessário).


In [2]:

# Se estiver no Colab e networkx não estiver instalado, descomente:
# !pip -q install networkx
import networkx as nx
from collections import deque, defaultdict
import math
import heapq
print("Ambiente OK: networkx, heapq, math")

Ambiente OK: networkx, heapq, math



---
## Parte 1 — Busca **não informada** (BFS e DFS)
**Objetivo:** modelar um problema como grafo, obter caminho via **BFS** (menor nº de arestas) e explorar **DFS** (ordem e um caminho possível).
    
**Teoria rápida**
- **BFS**: fronteira é **fila (FIFO)**. Em grafos **sem peso**, `nx.shortest_path` retorna caminho com menor número de arestas (equivale a BFS).
- **DFS**: fronteira é **pilha (LIFO)**. Pode não ser ótima; útil quando a memória é restrita.


In [3]:

# Grafo simples (não ponderado)
G = nx.Graph()
G.add_edges_from([
    ("A","B"),("A","C"),("B","D"),("C","D"),
    ("C","E"),("D","F"),("E","F"),("F","G")
])
print("Nós:", G.nodes())
print("Arestas:", G.edges())


Nós: ['A', 'B', 'C', 'D', 'E', 'F', 'G']
Arestas: [('A', 'B'), ('A', 'C'), ('B', 'D'), ('C', 'D'), ('C', 'E'), ('D', 'F'), ('E', 'F'), ('F', 'G')]


In [4]:

# BFS: menor número de arestas (em grafo não ponderado)
caminho_bfs = nx.shortest_path(G, source="A", target="G")
print("BFS (menor nº de passos):", caminho_bfs, "| passos:", len(caminho_bfs)-1)

BFS (menor nº de passos): ['A', 'B', 'D', 'F', 'G'] | passos: 4


In [5]:

# DFS: ordem de visita (pré-ordem) e um caminho possível via predecessores da DFS
ordem_dfs = list(nx.dfs_preorder_nodes(G, source="A"))
pre = nx.dfs_predecessors(G, source="A")
# reconstrói caminho até G (se existir)
caminho_dfs = ["G"]
while caminho_dfs[-1] in pre:
    caminho_dfs.append(pre[caminho_dfs[-1]])
caminho_dfs = list(reversed(caminho_dfs))
print("Ordem DFS:", ordem_dfs)
print("Um caminho via DFS:", caminho_dfs)

Ordem DFS: ['A', 'B', 'D', 'C', 'E', 'F', 'G']
Um caminho via DFS: ['A', 'B', 'D', 'C', 'E', 'F', 'G']



### Exercício 1 (BFS vs DFS)
1. **Mude a ordem** dos vizinhos (por exemplo, ordenando `G.neighbors(u)` alfabeticamente) e observe como **DFS** muda.  
2. Explique por que **BFS** mantém caminho mínimo (em nº de arestas) enquanto **DFS** pode não manter.



---
## Parte 2 — Grafos ponderados: **UCS (Dijkstra)**
**Objetivo:** comparar `shortest_path` **sem peso** (BFS) com `weight="weight"` (Dijkstra/UCS) em um grafo com custos.


In [6]:

# Grafo ponderado
WG = nx.Graph()
WG.add_weighted_edges_from([
    ("A","B", 2), ("A","C", 2),
    ("B","D", 5), ("C","D", 1),
    ("D","E", 2), ("E","F", 2), ("F","G", 3)
])
p_sem = nx.shortest_path(WG, "A", "G")  # ignora pesos
p_com = nx.shortest_path(WG, "A", "G", weight="weight")
c_com = nx.shortest_path_length(WG, "A", "G", weight="weight")
print("Sem peso (tipo BFS):", p_sem)
print("Com peso (UCS/Dijkstra):", p_com, "| custo:", c_com)

Sem peso (tipo BFS): ['A', 'B', 'D', 'E', 'F', 'G']
Com peso (UCS/Dijkstra): ['A', 'C', 'D', 'E', 'F', 'G'] | custo: 10



### Exercício 2 (Tráfego)
Aumente o peso da aresta `("C","D")` para simular trânsito pesado (ex.: `weight=50`) e observe a rota mudar.


In [7]:

WG_mod = WG.copy()
WG_mod["C"]["D"]["weight"] = 50  # trânsito pesado
p = nx.shortest_path(WG_mod, "A", "G", weight="weight")
c = nx.shortest_path_length(WG_mod, "A", "G", weight="weight")
print("Rota com trânsito:", p, "| custo:", c)

Rota com trânsito: ['A', 'B', 'D', 'E', 'F', 'G'] | custo: 14



---
## Parte 3 — Grid 2D com obstáculos: **Gulosa** e **A\***
**Objetivo:** construir um grid como grafo, usar **A\*** (f=g+h) com heurística **Manhattan**, e comparar com **busca gulosa** (só h).
    
**Heurísticas**
- `manhattan(a,b) = |ax-bx| + |ay-by|` (admissível/consistente para 4-direções com custo 1).
- `euclid(a,b) = √((ax-bx)^2 + (ay-by)^2)` (admissível; pode ser menos informativa em 4-direções, mas útil em 8-direções).


In [ ]:

def construir_grid_grafo(linhas, colunas, obstaculos=set()):
    Gg = nx.grid_2d_graph(linhas, colunas)  # 4 vizinhos
    Gg.remove_nodes_from(obstaculos)
    for u, v in Gg.edges():
        Gg[u][v]["weight"] = 1  # custo unitário
    return Gg

linhas, colunas = 6, 8
start, goal = (0,0), (5,7)
obst = {(0,5),(1,1),(1,2),(1,5),(2,7),(3,0),(3,1),(3,3),(4,3),(4,5),(5,1)}
GG = construir_grid_grafo(linhas, colunas, obstaculos=obst)

def manhattan(a,b):
    return abs(a[0]-b[0]) + abs(a[1]-b[1])
def euclid(a,b):
    return math.hypot(a[0]-b[0], a[1]-b[1])

# A* com Manhattan
caminho_astar = nx.astar_path(GG, start, goal, heuristic=manhattan, weight="weight")
print("A* (manhattan) caminho:", caminho_astar, "| custo:", len(caminho_astar)-1)

In [ ]:

def greedy_best_first(G, start, goal, h):
    pq = []
    heapq.heappush(pq, (h(start, goal), start))
    came = {start: None}
    visit = {start}
    exp = 0
    while pq:
        _, u = heapq.heappop(pq); exp += 1
        if u == goal:
            path = []
            while u is not None:
                path.append(u); u = came[u]
            return list(reversed(path)), exp
        for v in G.neighbors(u):
            if v not in visit:
                visit.add(v)
                came[v] = u
                heapq.heappush(pq, (h(v, goal), v))
    return None, exp

cam_g, ex_g = greedy_best_first(GG, start, goal, manhattan)
print("Gulosa:", cam_g, "| len:", len(cam_g)-1 if cam_g else None, "| expansões:", ex_g)

In [ ]:

def draw_path(path, title=""):
    s = set(path) if path else set()
    print(title)
    for r in range(linhas):
        row = []
        for c in range(colunas):
            cell = (r,c)
            if cell == start: ch = "S"
            elif cell == goal: ch = "G"
            elif cell in obst: ch = "█"
            elif cell in s: ch = "·"
            else: ch = " "
            row.append(ch)
        print("".join(row))
    print()

draw_path(cam_g, "Greedy Best-First")
draw_path(caminho_astar, "A* (Manhattan)")


### Exercício 3 (Parede enganosa)
Altere `obst` para criar uma **quase-parede** próxima de `goal` e compare **len(caminho)** e **expansões** da **Gulosa** vs **A\***. Explique por que a Gulosa pode piorar.


In [ ]:

# Exemplo de modificação rápida (experimente!)
obst2 = set(obst)
for r in range(1,5):
    obst2.add((r,6))  # quase-parede perto do goal (coluna 6)
GG2 = construir_grid_grafo(linhas, colunas, obstaculos=obst2)

cam_g2, ex_g2 = greedy_best_first(GG2, start, goal, manhattan)
cam_a2 = nx.astar_path(GG2, start, goal, heuristic=manhattan, weight="weight")
print("Greedy2: len", len(cam_g2)-1, "| exp", ex_g2)
print("A*2    : len", len(cam_a2)-1)



### Exercício 4 (Heurística)
Implemente `euclid` e use em `nx.astar_path`. Em 4-direções com custo 1, **Manhattan** costuma ser mais informativa; discuta.


In [ ]:

cam_e = nx.astar_path(GG, start, goal, heuristic=euclid, weight="weight")
print("A* com Euclidiana | len:", len(cam_e)-1)


### Exercício 5 (A* = UCS quando h=0)
Faça `h(n)=0` (troque a heurística por uma função que retorna zero). Observe que o algoritmo se comporta como **UCS**.


In [ ]:

zero = lambda a,b: 0
cam_zero = nx.astar_path(GG, start, goal, heuristic=zero, weight="weight")
print("A* com h=0 (UCS): len:", len(cam_zero)-1)


---
## Referências
- Russell, S.; Norvig, P. *Artificial Intelligence: A Modern Approach*, 4ª ed., 2021 — Cap. 3–4 (Busca).  
- Mitchell, T. *Machine Learning*, McGraw-Hill, 1997 — Formulação de problemas e conceitos básicos.  
- Géron, A. *Hands‑On Machine Learning with Scikit‑Learn, Keras & TensorFlow*, 2ª ed., O’Reilly, 2019 — Prática em Python.
